# 05e — Physical inventory processes: follow every unit

`ShelfLifeEngine` already handles the common perishable case: dated lots, FIFO use, and expiry. This notebook shows the next step. What if a recurring quality inspection removes stock, or a customer return adds it back? These are **physical flows** beyond ordinary demand, receipts, and expiry.

Stockcast calls a reusable rule for such a flow an **inventory process**. The process reports a named quantity at a defined time; the engine changes on-hand stock, records the flow, and checks the stock balance. We will first reproduce the familiar shelf-life result with the process API, then add one outflow and one inflow. Each step keeps the same two products and demand path.

All quantities and operating choices are synthetic. The aim is to understand where stock moves and which API to use.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from stockcast import InventoryStateDataFrame, OrderUpToPolicy, SimulationEngine
from stockcast.core import (
    Flow, InventoryProcess, ProcessFlows, ShelfLife, ShelfLifeEngine,
)
from stockcast.evaluation import validate_event_frame

%matplotlib inline

## 1. Declare one small chilled-inventory scenario

Yogurt and salad are reviewed and replenished daily with a one-day lead time. We generate 28 dated days of Poisson demand once with seed 7, then replay **exactly that table** in every run. The opening date is Sunday 1 March 2026, so demand period 0 is Monday 2 March. Unmet demand is lost sales rather than backlog.

The order-up-to levels of 18 and 26 are **fixed planner values** for this teaching example. They cover the two-day `L+R` decision window but are not estimated forecast quantiles or claims of a 90% service level. This notebook studies stock processes, not forecast selection.

In [ ]:
origin = pd.Timestamp("2026-03-01")          # a Sunday; demand starts on Monday
skus = ["yogurt", "salad"]
n_days = 28
rates = {"yogurt": 6.0, "salad": 9.0}

# One fixed demand path, shared by every run in this notebook.
rng = np.random.default_rng(7)
demand = pd.DataFrame([
    {"unique_id": sku, "period": day, "date": origin + pd.Timedelta(days=day + 1),
     "y": float(rng.poisson(rates[sku]))}
    for day in range(n_days) for sku in skus
])
demand.head()

The plot shows the realized daily demand that every branch will use. No process in this notebook changes these demand observations; it changes how much stock remains available to fulfill them.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.1), layout="constrained")
for sku, color in [("yogurt", "#167d9a"), ("salad", "#d58228")]:
    rows = demand.loc[demand["unique_id"].eq(sku)]
    ax.plot(rows["date"], rows["y"], marker="o", markersize=2.5,
            color=color, label=sku)
ax.set(title="One shared 28-day demand path", xlabel="Demand date",
       ylabel="Demand (kg per day)")
ax.legend(frameon=False)
plt.show()

In [ ]:
lead_time, review_period = 1, 1
# Fixed planner levels for the L+R window; no probability is asserted.
targets = pd.DataFrame({
    "unique_id": skus,
    "order_up_to": [18.0, 26.0],
    "target_end_date": origin + pd.Timedelta(days=lead_time + review_period),
})
policy = OrderUpToPolicy(
    lead_time=lead_time, review_period=review_period,
    service_level=None, allow_backorders=False,
).fit(
    targets, target_column="order_up_to",
    protection_horizon=lead_time + review_period, target_source="external_direct",
    forecast_origin=origin, forecast_frequency="D",
    target_end_date_column="target_end_date",
)

The stock count says how much is on hand; the lot table says **when those same units were received**. Yogurt begins with 5 kg from Saturday and 7 kg from Sunday, so its lot quantities add to the observed 12 kg. Salad's one Sunday lot adds to 15 kg. Stockcast checks this equality before simulating shelf life.

`run_options` holds the policy, demand, opening state, calendar, and scoring choices shared by every run below. We score all 28 days; there is no warm-up or settlement tail. The engine copies the opening state for each run.

In [ ]:
opening = pd.DataFrame({"unique_id": skus, "on_hand": [12.0, 15.0]})
inventory = InventoryStateDataFrame(
    skus, max_lead_time=lead_time, allow_backorders=False,
).initialize_from_observed(opening, on_hand_column="on_hand", start_date=origin)

opening_lots = pd.DataFrame({
    "unique_id": ["yogurt", "yogurt", "salad"],
    "received_date": [origin - pd.Timedelta(days=1), origin, origin],
    "quantity": [5.0, 7.0, 15.0],
})

# Every run below uses these same arguments.
run_options = dict(
    policy=policy, demand_source=demand, inventory=inventory, n_periods=n_days,
    period_frequency="D", warmup_periods=0, scoring_periods=n_days,
    settlement_periods=0, order_during_settlement=False,
    demand_source_name="synthetic chilled demand", random_seed=None,
)
opening_lots

## 2. The simple way: `ShelfLifeEngine`

Use this when expiry is the only extra physical mechanism. A lot received on day `D` with a three-day shelf life may serve demand on `D`, `D+1`, and `D+2`. It expires **before demand** on `D+3`. FIFO means demand consumes the oldest available lot first.

For example, the Saturday yogurt lot is old enough to expire on Tuesday; some of it may already have been sold by then. The event rows below show demand, receipts, fulfilled units, expiry, and end-of-day stock together. `expired_units` is a separate flow, not demand or lost sales.

In [ ]:
simple = ShelfLifeEngine(shelf_life_days=3).run(**run_options, opening_lots=opening_lots)
simple_events = simple.to_event_frame()
simple_events[["unique_id", "date", "demand", "received_units", "fulfilled_units",
               "expired_units", "ending_on_hand"]].head(8)

## 3. Write the same shelf-life run as a process

`ShelfLifeEngine` is the simple public route. Internally it uses a `ShelfLife` process on the standard `SimulationEngine`. The process form lets us add other stock flows later. We first run it **alone**, with the same three-day life and opening lots, then assert equality of every event row, column, and data type. There is no extra inspection or return yet.

In [ ]:
as_process = SimulationEngine().run(
    **run_options,
    processes=[ShelfLife(shelf_life_days=3, opening_lots=opening_lots)],
)
pd.testing.assert_frame_equal(as_process.to_event_frame(), simple_events)
print("Identical event ledgers; expired units:", simple_events["expired_units"].sum())

The physical results match, but the two entry points describe their settings differently. The familiar engine keeps its shelf-life settings; the process form lists each process under `run_settings["processes"]`. The entry below identifies its position, name, class, and one declared expiry outflow. This is provenance, not another stock transaction.

In [ ]:
entry = as_process.run_settings["processes"][0]
{key: entry[key] for key in ["position", "name", "class", "flows"]}

`to_process_flow_frame()` explains **which process produced a nonzero flow**. It has one row per SKU, period, and named flow with a nonzero quantity. For shelf life, the row says that `shelf_life.expired` removed stock before demand; those same units appear in the event ledger's `expired_units`. Days with no expiry have no flow row.

In [ ]:
as_process.to_process_flow_frame().head()

## 4. Add a Monday inspection outflow

Suppose staff check chilled products every Monday **after customers have been served**, discarding one quarter of what remains on the shelf, rounded down. This is neither a sale nor expiry, so it needs its own named physical outflow.

A custom `InventoryProcess` declares a unique `name`, a tuple of `Flow` definitions, and the hook for the time it acts. `Flow("discarded", "outflow")` says the returned quantity removes stock; it is nonnegative, because the direction belongs to the declaration. `after_demand(context)` sees the remaining on-hand amount as a read-only per-SKU Series. Returning `None` on other days means no change. Returning `ProcessFlows` asks the **engine** to apply and audit the discard; the process does not edit the state itself.

In [ ]:
class MondayInspection(InventoryProcess):
    """Each Monday a quality check discards a share of the stock on hand."""

    name = "inspection"
    flows = (Flow("discarded", "outflow"),)

    def __init__(self, share):
        self.share = share

    def after_demand(self, context):
        if context.date.day_name() != "Monday":
            return None                      # no flow today
        return ProcessFlows({"discarded": np.floor(self.share * context.on_hand)})

    def get_config(self):
        return {"share": self.share, "day": "Monday"}   # recorded in the manifest

Run the processes in an explicit order: `ShelfLife` first, then the inspection. At the start of each day shelf life expires old lots; after demand the inspection can remove some of the remaining stock. `ShelfLife` also mirrors the inspection's removal in its lot ledger by consuming the **oldest** lots. That oldest-first removal is a modelling choice; the example assumes inspected discards come from those lots.

The engine validates each declared flow and still owns every physical update. The next cell compares total expiry, inspection outflow, and inflow across all SKU-days.

In [ ]:
shelf_life = ShelfLife(shelf_life_days=3, opening_lots=opening_lots)
combined = SimulationEngine().run(
    **run_options, processes=[shelf_life, MondayInspection(share=0.25)],
)
events = validate_event_frame(combined.to_event_frame())
events[["expired_units", "process_outflow_units", "process_inflow_units"]].sum()

Because inspection introduces a general (non-expiry) flow, the event ledger gains `process_inflow_units` and `process_outflow_units`. The balance for one SKU-day is:

```text
ending_on_hand = starting_on_hand + received - backorders_fulfilled - fulfilled
                 - expired + inventory_adjustment
                 + process_inflow - process_outflow
```

Here there are no backorders, adjustments, or process inflows. Expiry remains in `expired_units`; inspection is in `process_outflow_units`, so they are **not counted twice**. The engine and `validate_event_frame()` both check this identity. The flow table names the process behind each nonzero amount.

In [ ]:
flows = combined.to_process_flow_frame()
print(flows.groupby(["process", "flow", "phase"])["quantity"].sum())
flows[flows["process"].eq("inspection")].head()

Look at the **first Monday** for yogurt. The two runs start with the same 12 kg and serve the same 6 kg of demand. The simple run ends with 6 kg. The inspection removes `floor(0.25 × 6) = 1` kg after demand, so the combined run ends with 5 kg. This one row shows where a general process outflow enters the accounting.

In [ ]:
first_monday = origin + pd.Timedelta(days=1)
columns = ["date", "starting_on_hand", "received_units", "fulfilled_units",
           "expired_units", "ending_on_hand"]
simple_row = simple_events.loc[
    simple_events["unique_id"].eq("yogurt") & simple_events["date"].eq(first_monday),
    columns,
].assign(scenario="Shelf life only", process_outflow_units=0.0)
combined_row = events.loc[
    events["unique_id"].eq("yogurt") & events["date"].eq(first_monday),
    columns + ["process_outflow_units"],
].assign(scenario="Shelf life + inspection")
pd.concat([simple_row, combined_row], ignore_index=True)[
    ["scenario", *columns, "process_outflow_units"]
]

In [ ]:
# The FIFO lots still add up to the stock on hand, product by product.
final_on_hand = combined.inventory.get_dataframe().set_index("unique_id")["on_hand"]
pd.DataFrame({"on_hand": final_on_hand,
              "lot_ledger": shelf_life.ledger.balances().reindex(final_on_hand.index)})

The simple shelf-life run loses 5 kg to expiry. With the Monday inspection, 9 kg is discarded and only 4 kg later expire; taking stock off the shelf earlier can reduce later expiry. In this particular path, both runs still sell the same kilograms. The chart shows only **unsold physical losses** by product and scenario; the accompanying table also reports sales, so we do not mistake a smaller expiry bar for better service.

In [ ]:
def outflows(result):
    rows = result.to_event_frame()
    columns = {"Sold": "fulfilled_units", "Expired": "expired_units"}
    if "process_outflow_units" in rows:
        columns["Discarded"] = "process_outflow_units"
    return pd.DataFrame({label: rows.groupby("unique_id")[column].sum()
                         for label, column in columns.items()})

comparison = pd.concat(
    {"Shelf life only": outflows(simple), "Shelf life + inspection": outflows(combined)},
    names=["scenario", "unique_id"],
).fillna(0.0)

losses = comparison[["Expired", "Discarded"]]
labels = [f"{sku} · {scenario}" for scenario, sku in losses.index]
fig, ax = plt.subplots(figsize=(8, 3.2), layout="constrained")
left = np.zeros(len(losses))
for column, color in [("Expired", "#B279A2"), ("Discarded", "#F58518")]:
    ax.barh(labels, losses[column], left=left, color=color, label=column)
    left += losses[column].to_numpy()
ax.invert_yaxis()
ax.set(xlabel="Units lost over 28 days", title="Stock that left the shelf unsold")
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
comparison

The totals hide timing. For yogurt, the next plot shows end-of-day on-hand under the two runs and the days when inspection removed stock. Monday's orange bars are **after-demand** discards, so they affect the closing stock and later periods. The purple markers show expiry on the combined run, which happens at the **start** of its date. This helps distinguish the two different kinds of unsold loss.

In [ ]:
yogurt_simple = simple_events.loc[simple_events["unique_id"].eq("yogurt")]
yogurt_combined = events.loc[events["unique_id"].eq("yogurt")]
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, layout="constrained")
axes[0].step(yogurt_simple["date"], yogurt_simple["ending_on_hand"], where="post",
             color="#167d9a", label="Shelf life only")
axes[0].step(yogurt_combined["date"], yogurt_combined["ending_on_hand"], where="post",
             color="#d58228", label="With Monday inspection")
axes[0].set(ylabel="End-of-day on hand (kg)", title="Yogurt: inspection changes when stock leaves")
axes[0].legend(frameon=False)
axes[1].bar(yogurt_combined["date"], yogurt_combined["process_outflow_units"],
            width=0.7, color="#d58228", label="Discarded after demand")
expired = yogurt_combined.loc[yogurt_combined["expired_units"].gt(0)]
axes[1].scatter(expired["date"], expired["expired_units"],
                color="#7b519d", marker="D", s=44, label="Expired before demand", zorder=3)
axes[1].set(ylabel="Unsold loss (kg)", xlabel="Demand date")
axes[1].legend(frameon=False)
plt.show()

## 5. An inflow needs an honest lot date

Now suppose 3 kg of salad are returned before demand on Wednesday 11 March. A `before_demand` process can add them to on-hand stock before the policy decision and customers arrive. But once shelf life is active, returned units must enter a dated lot. Without `received_dates`, Stockcast cannot know their age or expiry date and rejects the run. The next cell demonstrates that fail-closed error deliberately.

In [ ]:
class CustomerReturns(InventoryProcess):
    """Scheduled returns, put back on the shelf before the day's demand."""

    name = "returns"
    flows = (Flow("returned", "inflow"),)

    def __init__(self, schedule, lot_date=None):
        self.schedule = schedule        # {date: {product: units}}
        self.lot_date = lot_date        # a function of the current date, or None

    def before_demand(self, context):
        units = self.schedule.get(context.date)
        if units is None:
            return None
        if self.lot_date is None:
            return ProcessFlows({"returned": units})
        return ProcessFlows({"returned": units},
                            received_dates={"returned": self.lot_date(context.date)})


return_day = origin + pd.Timedelta(days=10)
try:
    SimulationEngine().run(**run_options, processes=[
        ShelfLife(3, opening_lots), CustomerReturns({return_day: {"salad": 3.0}}),
    ])
except ValueError as error:
    print("Rejected:", error)
else:
    raise AssertionError("expected shelf life to reject an undated return")

For the successful version, we **declare** that the returned salad is one day old: its lot date is the day before it re-enters the shop. This is a teaching assumption, not an age Stockcast infers. `received_dates={"returned": ...}` gives shelf life the provenance it needs. `process_inflow_units` records the 3 kg on the return day; because this occurs before receipts, ordering, and demand, the added stock is visible to that day's decision and can serve customers.

In [ ]:
returns = CustomerReturns({return_day: {"salad": 3.0}},
                          lot_date=lambda date: date - pd.Timedelta(days=1))
with_returns = SimulationEngine().run(**run_options, processes=[
    ShelfLife(3, opening_lots), MondayInspection(share=0.25), returns,
])
return_rows = validate_event_frame(with_returns.to_event_frame())
return_rows.loc[return_rows["date"].eq(return_day),
                ["unique_id", "date", "process_inflow_units", "fulfilled_units",
                 "ending_on_hand"]]

## Choosing the right extension point

| Operating need | API to use |
|---|---|
| FIFO expiry with one shelf life | `ShelfLifeEngine(shelf_life_days=...)` |
| Recurring physical inflows or outflows alongside shelf life | `SimulationEngine().run(..., processes=[ShelfLife(...), YourProcess()])` |
| One dated correction to observed stock | `ScheduledInventoryAdjustment` callback (Notebook 08) |
| One accepted order delivered in planned parts | `SupplyModel` partial deliveries (Notebook 05d) |

A process may propose an on-hand change in `before_demand` or `after_demand`; `on_receipt` lets it **observe** receipts. Processes cannot directly change the pipeline, backlog, or demand. Partial deliveries schedule the full order in parts; they do not model a permanent supplier shortfall after placement. See the [Physical processes guide](../../../docs/guides/physical-processes.md) for the full contract.